<a href="https://colab.research.google.com/github/SWATHIRAVIRS/fake_news_detection/blob/main/fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q transformers datasets evaluate accelerate

In [5]:
!unzip data.csv.zip

Archive:  data.csv.zip
replace data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: data.csv                


In [6]:
#cleaning
import pandas as pd
import torch
import evaluate
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
df=pd.read_csv("/content/data.csv.zip")

In [7]:
df.head()

,URLs,Headline,Body,Label
0,http://www.bbc.com/news/world-us-canada-414191...,Four ways Bob Corker skewered Donald Trump,Image copyright Getty Images\nOn Sunday mornin...,1
1,https://www.reuters.com/article/us-filmfestiva...,Linklater's war veteran comedy speaks to moder...,"LONDON (Reuters) - “Last Flag Flying”, a comed...",1
2,https://www.nytimes.com/2017/10/09/us/politics...,Trump’s Fight With Corker Jeopardizes His Legi...,The feud broke into public view last week when...,1
3,https://www.reuters.com/article/us-mexico-oil-...,Egypt's Cheiron wins tie-up with Pemex for Mex...,MEXICO CITY (Reuters) - Egypt’s Cheiron Holdin...,1
4,http://www.cnn.com/videos/cnnmoney/2017/10/08/...,Jason Aldean opens 'SNL' with Vegas tribute,"Country singer Jason Aldean, who was performin...",1


In [8]:
type(df)

pandas.core.frame.DataFrame

In [9]:
df['text']=df["Headline"] + " " + df["Body"]

In [10]:
df=df[["text","Label"]]

In [11]:
df.head()

,text,Label
0,Four ways Bob Corker skewered Donald Trump Ima...,1
1,Linklater's war veteran comedy speaks to moder...,1
2,Trump’s Fight With Corker Jeopardizes His Legi...,1
3,Egypt's Cheiron wins tie-up with Pemex for Mex...,1
4,Jason Aldean opens 'SNL' with Vegas tribute Co...,1


In [12]:
df["Label"].value_counts()

,count
Label,
0,2137
1,1872


In [13]:
df.isnull().sum()

,0
text,21
Label,0


In [14]:
df.duplicated().sum()

np.int64(485)

In [15]:
df.drop_duplicates(inplace=True)

In [16]:
df=df.dropna()

In [17]:
df.isnull().sum()

,0
text,0
Label,0


In [18]:
df=df.reset_index(drop=True)

In [19]:
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42)

In [20]:
from datasets import Dataset
train_dataset=Dataset.from_pandas(train_df.drop(columns=['__index_level_0__'],errors='ignore'))
test_dataset=Dataset.from_pandas(test_df.drop(columns=['__index_level_0__'],errors='ignore'))

In [21]:
#tokenzier
model_checkpoint="distilbert-base-uncased"
tokenzier=AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
#tokenization
def preprocess_function(examples):
  result = tokenzier(
      examples["text"],
      truncation=True,
      padding=True,
      max_length=512
  )
  result["labels"]=examples["Label"]
  return result

In [23]:
tokenized_train=train_dataset.map(preprocess_function,batched=True,remove_columns=train_dataset.column_names)
tokenized_test=test_dataset.map(preprocess_function,batched=True,remove_columns=test_dataset.column_names)

Map:   0%|          | 0/2817 [00:00<?, ? examples/s]

Map:   0%|          | 0/705 [00:00<?, ? examples/s]

In [24]:
# Rename 'Label' to 'labels' so the model can find it to calculate loss
tokenized_train = tokenized_train.set_format("torch",columns=["input_ids","attention_mask", "labels"])
tokenized_test = tokenized_test.set_format("torch",columns=["input_ids","attention_mask", "labels"])

In [25]:
#model selection
from transformers import AutoModelForSequenceClassification
model=AutoModelForSequenceClassification.from_pretrained(model_checkpoint,num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=tokenzier)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
import numpy as np
metrices=evaluate.load("accuracy")
def compute_metrices(eval_pred):
  logits,label = eval_pred
  predictions=np.argmax(logits,axis=1)
  return metrices.compute(predictions=predictions,references=label)

In [31]:
#training setup
from transformers import TrainingArguments
training_args=TrainingArguments(
    output_dir="/fake_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

In [32]:
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model is running on:{device}")

Model is running on:cuda


In [35]:
def preprocess_function(examples):
    result = tokenzier(examples["text"], truncation=True, padding="max_length", max_length=512)
    result["labels"] = examples["Label"]
    return result

final_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
final_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

# 2. Force the Trainer to use these new variables
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train,  # Using the 'final_' versions we just made
    eval_dataset=final_test,    # This WILL NOT be None anymore
    data_collator=DataCollatorWithPadding(tokenizer=tokenzier),
    compute_metrics=compute_metrices,
)

print(f"Dataset ready! Training on {len(final_train)} samples...")
trainer.train()

Map:   0%|          | 0/2817 [00:00<?, ? examples/s]

Map:   0%|          | 0/705 [00:00<?, ? examples/s]

Dataset ready! Training on 2817 samples...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.054643,0.990071
2,No log,0.059007,0.990071


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=354, training_loss=0.01638075726180427, metrics={'train_runtime': 100.7101, 'train_samples_per_second': 55.943, 'train_steps_per_second': 3.515, 'total_flos': 746321324027904.0, 'train_loss': 0.01638075726180427, 'epoch': 2.0})

In [37]:
from transformers import pipeline
news_detector = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenzier,
    device=0 if torch.cuda.is_available() else -1
)

def check_news(text):
    result = news_detector(text)[0]
    label = result['label']
    score = result['score']
    status = "REAL" if label == "LABEL_1" else "FAKE"

    print(f"Prediction: {status}")
    print(f"Confidence: {score:.2%}")

my_headline = "NASA discovers life on Mars inside a chocolate bar."
check_news(my_headline)

Prediction: REAL
Confidence: 99.48%
